In [1]:
import numpy as np
import torch
from torch.utils import data

### 3.3.1 生成数据集

In [2]:
def synthetic_data(w, b, num_examples):     #@save
    """生成数据集(y=Xw+b+噪声)"""
    # 使用torch.normal()正态分布函数生成符合正态分布的随机数
    # 均值为0，标准差为1，输出张量形状为(num_examples, len(w))
    X = torch.normal(0, 1, (num_examples, len(w)))
    # 使用torch.matmul(X, w)计算矩阵乘法，再加上偏置(bias): b
    y = torch.matmul(X, w) + b
    # 添加随机噪声，使用torch.normal()函数生成符合正态分布的随机噪声值
    # 均值为0，标准差为0.01，输出张量形状与y的形状一致
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

In [3]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

### 3.3.2 读取数据集

In [6]:
def load_array(data_arrays, batch_size, is_train=True):
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train, num_workers=0)

In [7]:
batch_size = 10
data_iter = load_array((features, labels), batch_size)

In [8]:
next(iter(data_iter))

[tensor([[ 1.1375, -0.3175],
         [ 0.3268, -0.7306],
         [-1.8451, -0.3447],
         [ 1.9604, -0.5441],
         [ 1.0516, -0.2319],
         [ 0.3711,  1.0228],
         [-0.4526, -0.6323],
         [ 0.5394, -0.8298],
         [-0.3648,  2.4318],
         [ 0.4524,  0.1921]]),
 tensor([[ 7.5644],
         [ 7.3572],
         [ 1.6869],
         [ 9.9743],
         [ 7.0967],
         [ 1.4788],
         [ 5.4338],
         [ 8.0925],
         [-4.7992],
         [ 4.4568]])]

### 3.3.3 定义模型

In [9]:
# nn是神经网络的缩写
from torch import nn
net = nn.Sequential(nn.Linear(2, 1))

### 3.3.4 初始化模型参数

In [11]:
net[0].weight.data.normal_(0.0, 0.01)       # 权重参数初始化为从均值为0，标准差为0.01的正态分布中随机抽样
net[0].bias.data.fill_(0)                   # 偏置参数初始化为0

tensor([0.])

### 3.3.5 定义损失函数

In [12]:
loss = nn.MSELoss()                     # 定义损失函数为计算均方误差，使用MSELoss类

### 3.3.6 定义优化算法

In [13]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03, momentum=0.9, weight_decay=0.0001)

### 3.3.7 训练

In [14]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.002129
epoch 2, loss 0.000106
epoch 3, loss 0.000103


In [15]:
w = net[0].weight.data
print('w的估计误差: ', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差: ', true_b - b)

w的估计误差:  tensor([ 0.0004, -0.0008])
b的估计误差:  tensor([0.0009])
